# 模块 B 综合项目：sklearn 分类完整工作流

使用含缺失值和类别列的模拟客户数据，完成预处理、分类训练、评价与模型/图表交付。

**项目类型：**教师/自学参考答案 · 分步骤完整代码

## 提交要求

1. 生成客户表并拆分 X/y
2. 训练集/测试集分层划分，数值和类别列分别预处理
3. Pipeline 训练逻辑回归与测试预测
4. 输出准确率、F1、混淆矩阵和图表
5. 保存并重新加载 joblib 模型

## 0. 准备工作：执行以下代码生成本地素材

这个单元格可以直接运行。素材为合成测试数据，无需联网。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix

out=Path("B_results");out.mkdir(exist_ok=True)
rng=np.random.default_rng(42)
n=160
age=rng.integers(18,60,size=n).astype(float)
score=rng.normal(60,18,size=n)
region=rng.choice(["东区","西区"],size=n)
target=((score+age*0.25+(region=="东区")*7+rng.normal(0,10,n))>72).astype(int)
age[::15]=np.nan
df=pd.DataFrame({"Age":age,"Score":score,"Region":region,"Pass":target})
print("数据:",df.shape,"标签:",df["Pass"].value_counts().to_dict())

## TODO 1：训练集/测试集分层划分，数值和类别列分别预处理

运行参考实现，观察变量变化与输出。 

In [ ]:
X=df.drop(columns="Pass");y=df["Pass"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
num=Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())])
cat=Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))])
pre=ColumnTransformer([("num",num,["Age","Score"]),("cat",cat,["Region"])])
model=Pipeline([("pre",pre),("model",LogisticRegression(max_iter=500))])
model.fit(X_train,y_train)
prediction=model.predict(X_test)

## TODO 2：Pipeline 训练逻辑回归与测试预测

运行参考实现，观察变量变化与输出。 

In [ ]:
acc=accuracy_score(y_test,prediction)
f1=f1_score(y_test,prediction,zero_division=0)
matrix=confusion_matrix(y_test,prediction,labels=[0,1])
print("Accuracy:",round(acc,3),"F1:",round(f1,3))
print("混淆矩阵:\n",matrix)

## TODO 3：输出准确率、F1、混淆矩阵和图表

运行参考实现，观察变量变化与输出。 

In [ ]:
pd.DataFrame({"y_true":y_test.to_numpy(),"y_pred":prediction}).to_csv(out/"B_predictions.csv",index=False)
joblib.dump(model,out/"B_classifier.pkl")
fig,ax=plt.subplots()
ax.imshow(matrix,cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j,i,str(matrix[i,j]),ha="center",va="center")
ax.set_xticks([0,1]);ax.set_yticks([0,1])
ax.set_xlabel("Predicted");ax.set_ylabel("Actual");ax.set_title("Confusion matrix")
fig.tight_layout();fig.savefig(out/"B_confusion.png",dpi=150);plt.close(fig)

## TODO 4：保存并重新加载 joblib 模型

运行参考实现，观察变量变化与输出。 

In [ ]:
loaded=joblib.load(out/"B_classifier.pkl")
assert np.array_equal(loaded.predict(X_test),prediction)
for name in ["B_predictions.csv","B_classifier.pkl","B_confusion.png"]:
    assert (out/name).is_file() and (out/name).stat().st_size>0,name
print("B 模块全部交付文件核验通过")

## 交付核验

- `B_results/B_predictions.csv`
- `B_results/B_classifier.pkl`
- `B_results/B_confusion.png`

完成后核对所有文件是否存在并可打开。参考答案在 `solutions/B_project_solution.ipynb`。

In [ ]:
# 交付文件自检：仅检查文件是否存在且非空，不自动判断模型效果
from pathlib import Path
folder=Path("B_results")
required=["B_predictions.csv","B_classifier.pkl","B_confusion.png"]
missing=[name for name in required if not (folder/name).is_file() or (folder/name).stat().st_size==0]
if missing:
    print("缺失或为空的文件：",missing)
else:
    print("全部交付文件已生成")
